In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Pengoptimum Portfolio Kuantum: Fungsi Qiskit oleh Global Data Quantum


> **Note:** Fungsi Qiskit ialah ciri eksperimental yang hanya tersedia untuk pengguna Pelan Premium, Pelan Flex, dan Pelan On-Prem IBM Quantum&reg; (melalui API IBM Quantum Platform). Ciri ini dalam status keluaran pratonton dan tertakluk kepada perubahan.
# Gambaran Keseluruhan
Pengoptimum Portfolio Kuantum ialah Fungsi Qiskit yang menangani masalah pengoptimuman portfolio dinamik — masalah piawai dalam kewangan yang bertujuan untuk mengimbangi semula pelaburan berkala merentas set aset bagi memaksimumkan pulangan dan meminimumkan risiko. Dengan menggunakan teknik pengoptimuman kuantum terkini, fungsi ini memudahkan proses supaya pengguna tanpa kepakaran dalam pengkomputeran kuantum dapat memanfaatkan kelebihannya dalam mencari trajektori pelaburan yang optimum. Sesuai untuk pengurus portfolio, penyelidik dalam kewangan kuantitatif, dan pelabur individu, alat ini membolehkan pengujian semula strategi perdagangan dalam pengoptimuman portfolio.
# Penerangan Fungsi
Fungsi Pengoptimum Portfolio Kuantum menggunakan algoritma Variational Quantum Eigensolver (VQE) untuk menyelesaikan masalah Quadratic Unconstrained Binary Optimization (QUBO) bagi menangani masalah pengoptimuman portfolio dinamik. Pengguna hanya perlu menyediakan data harga aset dan mentakrifkan kekangan pelaburan, kemudian fungsi menjalankan proses pengoptimuman kuantum yang mengembalikan set trajektori pelaburan yang telah dioptimumkan.

Proses ini terdiri daripada empat peringkat utama. Pertama, data input dipetakan kepada masalah yang serasi dengan kuantum, membina QUBO bagi masalah pengoptimuman portfolio dinamik, dan mengubahnya menjadi operator kuantum (Ising Hamiltonian). Seterusnya, masalah input dan algoritma VQE disesuaikan untuk dijalankan pada perkakasan kuantum. Algoritma VQE kemudiannya dijalankan pada perkakasan kuantum, dan akhirnya, keputusan diproses semula untuk menyediakan trajektori pelaburan yang optimum. Sistem ini juga menyertakan pemprosesan semula yang peka hingar (berasaskan [SQD](/guides/qiskit-addons-sqd)) untuk memaksimumkan kualiti output.

Fungsi Qiskit ini berdasarkan [manuskrip yang diterbitkan](https://arxiv.org/abs/2412.19150) oleh Global Data Quantum.
![Visualisasi aliran kerja fungsi](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)
# Input
Argumen input fungsi diterangkan dalam jadual berikut. Data aset dan spesifikasi masalah lain mesti disediakan; selain itu, tetapan VQE boleh disertakan untuk menyesuaikan proses pengoptimuman.
| **Nama**               | **Jenis**          | **Penerangan**                                           | **Diperlukan** | **Lalai**                            | **Contoh**                               |
|------------------------|--------------------|----------------------------------------------------------|----------------|--------------------------------------|------------------------------------------|
| assets                 | json               | Kamus dengan harga aset                                  | Ya             | -                                    | -                                        |
| qubo_settings          | json               | Tetapan QUBO                                             | Ya             | -                                    | Lihat contoh dalam jadual di bawah       |
| ansatz_settings        | json               | Tetapan ansatz                                           | Tidak          | `None`                               | Lihat contoh dalam jadual di bawah.      |
| optimizer_settings     | json               | Tetapan pengoptimum                                      | Tidak          | `None`                               | Lihat contoh dalam jadual di bawah.      |
| backend                | str                | Nama Backend QPU                                         | Tidak          | -                                    | `"ibm_torino"`                           |
| previous_session_id    | list of str        | Senarai ID sesi untuk mendapatkan semula data daripada larian sebelumnya[(*)](#1-note) | Tidak | Senarai kosong | `["session_id_1", "session_id_2"]` |
| apply_postprocess      | bool               | Guna pemprosesan semula SQD peka hingar                  | Tidak          | `True`                               | `True`                                   |
| tags                   | list of strings    | Senarai tag untuk mengenal pasti eksperimen              | Tidak          | Senarai kosong                       | `["optimization", "quantum_computing"]`  |
<span id="1-note"></span>
*Untuk menyambung semula pelaksanaan atau mendapatkan semula kerja yang diproses dalam satu atau lebih sesi sebelumnya, senarai ID sesi mesti dihantar dalam parameter `previous_session_id`. Ini amat berguna dalam kes di mana tugas pengoptimuman gagal selesai akibat ralat dalam proses, dan pelaksanaan perlu diteruskan. Untuk mencapai ini, anda mesti menyediakan argumen yang sama yang digunakan dalam pelaksanaan awal, bersama senarai `previous_session_id` seperti yang diterangkan.
> **Caution:** Memuatkan data untuk sesi sebelumnya (untuk menyambung semula pengoptimuman) boleh mengambil masa sehingga satu jam.
## `assets`
Data mesti disusun sebagai objek JSON yang menyimpan maklumat tentang harga penutup aset kewangan pada tarikh tertentu. Formatnya adalah seperti berikut:

- Kunci utama (string): Nama atau simbol ticker aset kewangan (contohnya, "8801.T").
- Kunci sekunder (string): Tarikh dalam format YYYY-MM-DD.
- Nilai (nombor): Harga penutup aset pada tarikh yang dinyatakan. Harga boleh dimasukkan sama ada dalam bentuk ternormal atau tidak ternormal.

*Perhatikan bahawa semua kamus mesti mempunyai kunci sekunder yang sama (tarikh). Jika sesebuah aset tidak mempunyai tarikh yang dimiliki aset lain, data mesti diisi untuk memastikan konsistensi. Contohnya, ini boleh dilakukan dengan menggunakan harga penutup terakhir yang dicatat bagi aset tersebut.*
### Contoh
``` py
{
    "8801.T": {
        "2023-01-01": 2374.0,
        "2023-01-02": 2374.0,
        "2023-01-03": 2374.0,
        "2023-01-04": 2356.5,
        ...
    },
    "AAPL": {
        "2023-01-01": 145.2,
        "2023-01-02": 146.5,
        "2023-01-03": 147.3,
        "2023-01-04": 148.1,
        ...
    },
    ...
}
```

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

> **Note:** Data aset mesti mengandungi, sekurang-kurangnya, harga penutup pada ``(nt+1) * dt`` (lihat bahagian input [`qubo_settings`](#qubo_settings)) cap masa (contohnya, hari).
## `qubo_settings`
Jadual seterusnya menerangkan kunci kamus `qubo_settings`. Bina kamus dengan menentukan bilangan langkah masa `nt`, bilangan qubit resolusi `nq`, dan `max_investment` — atau ubah nilai lalai lain.
| Nama                | Jenis   | Penerangan                                                                  | Diperlukan | Lalai | Contoh |
|---------------------|---------|-----------------------------------------------------------------------------|------------|-------|--------|
| nt                  | int     | Bilangan langkah masa                                                        | Ya         | -     | 4      |
| nq                  | int     | Bilangan Qubit resolusi                                                      | Ya         | -     | 4      |
| max_investment      | float   | Bilangan maksimum unit mata wang yang dilaburkan merentasi semua aset        | Ya         | -     | 10     |
| dt*                  | int     | Tetingkap masa yang dipertimbangkan dalam setiap langkah masa. Unitnya sepadan dengan selang masa antara kunci dalam data aset | Tidak | 30 | - |
| risk_aversion       | float   | Pekali pengelakan risiko                                                     | Tidak      | 1000  | -      |
| transaction_fee     | float   | Pekali yuran transaksi                                                       | Tidak      | 0.01  | -      |
| restriction_coeff   | float   | Pendarab Lagrange yang digunakan untuk menguatkuasakan kekangan masalah dalam formulasi QUBO | Tidak | 1 | - |
## `ansatz_settings`
Untuk mengubah pilihan lalai, cipta kamus untuk parameter `ansatz_settings` dengan kunci berikut. Secara lalai, ansatz ditetapkan kepada `"real_amplitudes"`, dan kedua-dua pilihan tambahan (lihat jadual berikut) ditetapkan kepada `False`.
| Nama                  | Jenis    | Penerangan                                                                  | Diperlukan | Lalai |
|-----------------------|----------|-----------------------------------------------------------------------------|------------|-------|
| ansatz[*](#single-star)               | str      | Ansatz yang akan digunakan                                                  | Tidak      | `"real_amplitudes"` |
| multiple_passmanager[**](#double-star) | bool    | Mendayakan subrutin passmanager berganda (tidak tersedia untuk ansatz Tailored) | Tidak | `False` |
| dd_enable   | bool     | Menambah dynamical decoupling                                                | Tidak      | `False` |

<span id="single-star"></span>
\* Ansatz yang tersedia
- `real_amplitudes`
- `cyclic`
- `optimized_real_amplitudes`
- `tailored` (Hanya untuk Backend `ibm_torino`, 7 aset, 4 langkah masa, dan 4 qubit resolusi)

<span id="double-star"></span>
** Jika ``multiple_passmanager`` ditetapkan kepada ``False``, fungsi menggunakan pass manager Qiskit lalai dengan `optimization_level=3`. Jika ditetapkan kepada ``True``, subrutin ``multiple_passmanager`` membandingkan tiga pass manager: pass manager Qiskit lalai sebelumnya, pass manager yang memetakan qubit ke atas rantaian jiran terdekat QPU, dan perkhidmatan [Transpiler AI](/guides/ai-transpiler-passes). Kemudian, pass manager dengan anggaran ralat kumulatif yang lebih rendah dipilih.
## `optimizer_settings`
Parameter ini ialah kamus dengan beberapa pilihan yang boleh dilaraskan bagi proses pengoptimuman.
| Nama               | Jenis  | Penerangan                                      | Diperlukan | Lalai                      |
|--------------------|--------|-------------------------------------------------|------------|----------------------------|
| primitive_options  | json   | Tetapan primitif                                | Tidak      | -                          |
| optimizer          | str    | Pengoptimum klasik yang dipilih                 | Tidak      | `"differential_evolution"` |
| optimizer_options  | json   | Konfigurasi pengoptimum                         | Tidak      | -                          |
> **Note:** Pada masa ini, satu-satunya pilihan pengoptimum yang tersedia ialah `"differential_evolution"`.

Di bawah kunci `primitive_options` dan `optimizer_options`, kita tetapkan kamus dengan parameter berikut:
### `primitive_options`
| Nama              | Jenis  | Penerangan                                 | Diperlukan | Lalai                      | Contoh                     |
|-------------------|--------|--------------------------------------------|------------|----------------------------|----------------------------|
| sampler_shots     | int    | Bilangan shot Sampler.                     | Tidak      | 100000                     | -                          |
| estimator_shots   | int    | Bilangan shot Estimator.                   | Tidak      | 25000                      | -                          |
| estimator_precision | float | Ketepatan yang diingini bagi nilai jangkaan. Jika dinyatakan, ketepatan akan digunakan sebagai ganti `estimator_shots`. | Tidak | `None` | 0.015625 · (1 / sqrt(4096)) |
| max_time          | int or str | Jumlah masa maksimum sesi runtime boleh kekal terbuka sebelum ditutup secara paksa. Boleh diberikan dalam saat (int) atau sebagai string, seperti `"2h 30m 40s"`. Mesti kurang daripada had maksimum sistem. | Tidak | `None` | `"1h 15m"` |
### `optimizer_options`
| Nama              | Jenis    | Penerangan                               | Diperlukan | Lalai         |
|-------------------|----------|------------------------------------------|------------|---------------|
| num_generations   | int      | Bilangan generasi                        | Tidak      | 20            |
| population_size   | int      | Saiz populasi                            | Tidak      | 20            |
| mutation_range    | list     | Faktor mutasi maksimum dan minimum       | Tidak      | [0, 0.25]     |
| recombination     | float    | Faktor rekombinasi                       | Tidak      | 0.4           |
| max_parallel_jobs | int      | Bilangan maksimum kerja QPU yang dilaksanakan secara selari | Tidak | 3 |
| max_batchsize     | int      | Saiz kelompok maksimum                   | Tidak      | 200           |
> **Note:** - Bilangan generasi yang dinilai oleh evolusi pembezaan ialah `num_generations` + 1 kerana populasi awal disertakan.
> - Jumlah bilangan Circuit dikira sebagai `(num_generations + 1) * population_size`.
> - Menggunakan saiz populasi yang lebih besar dan lebih banyak generasi secara amnya meningkatkan kualiti hasil pengoptimuman. Walau bagaimanapun, tidak disyorkan untuk melebihi saiz populasi 120 dan bilangan generasi melebihi 20 (contohnya, ``120 * 21 = 2520`` jumlah Circuit), kerana ini akan menjana bilangan Circuit yang berlebihan, yang boleh menjadi mahal dari segi pengiraan dan memakan masa.
> 
> - Fungsi ini membolehkan anda menyambung semula pengoptimuman sebelumnya, dan sentiasa boleh meningkatkan bilangan generasi (dengan menyediakan input yang sama kecuali untuk `previous_session_id` dan ``num_generations`` yang ditingkatkan).
> **Note:** Pastikan pematuhan dengan had kerja Qiskit Runtime.
> 
> - Sampler: `sampler_shots <= 10_000_000`.
> - Estimator: `max_batchsize * estimator_shots * observable_size <= 10_000_000` (untuk fungsi ini, semua sebutan observable bertukar, jadi `observable_size=1`).
> 
> Lihat panduan [Had Kerja](/guides/job-limits) untuk maklumat lanjut.
# Output

Fungsi mengembalikan dua kamus: kamus `"result"`, yang mengandungi hasil pengoptimuman terbaik, termasuk penyelesaian optimum dan kos objektif minimum yang berkaitan; dan `"metadata"`, dengan data daripada semua hasil yang diperoleh semasa proses pengoptimuman, berserta metrik masing-masing.

Kamus pertama memberi tumpuan kepada penyelesaian yang berprestasi terbaik, manakala yang kedua menyediakan maklumat terperinci tentang semua penyelesaian, termasuk kos objektif dan metrik lain yang relevan.

## Kamus Output:
| **Nama**    | **Jenis**                    | **Penerangan**                                                                  | **Contoh**  |
|-------------|------------------------------|---------------------------------------------------------------------------------|-------------|
| result      | dict[str, dict[str, float]]  | Mengandungi strategi pelaburan dari masa ke masa, dengan setiap cap masa dipetakan kepada wajaran pelaburan khusus aset (setiap wajaran ialah jumlah pelaburan yang dinormalkan oleh jumlah pelaburan keseluruhan). | `{'time_1': {'asset_1': 0.2, 'asset_2': 0.3, ...}, ...}` |
| metadata    | dict[str, Any]               | Data yang dijana semasa analisis, termasuk penyelesaian, kos, dan metrik.        | Lihat contoh di bawah |
### Penerangan kamus `metadata`
| **Nama**                 | **Jenis**             | **Penerangan**                                                                                     | **Contoh**                   |
|--------------------------|-----------------------|----------------------------------------------------------------------------------------------------|------------------------------|
| session_id               | str                   | Pengenal pasti unik untuk sesi IBM Quantum.                             | `"d0h30qjvpqf00084fgw0"`     |
| all_samples_metrics | dict                 | Kamus yang mengandungi pelbagai metrik untuk setiap sampel yang diproses semula, seperti kos atau kekangan. | Lihat penerangan di bawah |
| sampler_counts           | dict[str, int]        | Kamus di mana kunci ialah representasi bitstring bagi penyelesaian yang disampel dan nilai ialah kiraan mereka. | `{"101010": 3, "111000": 1}` |
| asset_order              | list[str]             | Senarai dengan tertib pelaburan aset yang sepadan pada setiap langkah masa dalam strategi pelaburan. | `["Asset_0", "Asset_1", "Asset_3"]` |
| QUBO                     | list[list[float]]     | Matriks QUBO bagi masalah.                                                                          | `[[-6.96e-01, 5.81e-01, -1.26e-02, 0.00e+00], ...]` |
| resource_summary           | dict[str, dict[str, float]] | Ringkasan masa penggunaan CPU dan QPU (dalam saat) merentasi peringkat proses yang berbeza. | `{'RUNNING: EXECUTING_QPU': {'CPU_TIME': 412.84, 'QPU_TIME': 87.22}, ...}` |
### Penerangan kamus `all_samples_metrics`
| **Nama**                | **Jenis**      | **Penerangan**                                                                                                   | **Contoh**                   |
|-------------------------|----------------|------------------------------------------------------------------------------------------------------------------|------------------------------|
| investment_trajectories | list[list]     | Strategi pelaburan yang diperoleh daripada keadaan kuantum yang dinyahkod. | `[[1, 2, 2], [1, 2, 1]]`     |

| counts                  | list[int]      | Bilangan kali setiap trajektori pelaburan disampel. Indeks sepadan dengan `investment_trajectories`.             | `[5, 3]`                     |
| objective_costs         | list[float]    | Nilai fungsi objektif bagi setiap trajektori pelaburan, disusun dari terendah ke tertinggi.                      | `[0.98, 1.25]`               |
| sharpe_ratios           | list[float]    | Prestasi terlaras risiko (nisbah Sharpe) bagi setiap trajektori pelaburan. Diselaraskan mengikut indeks.         | `[1.1, 0.7]`                 |
| returns                 | list[float]    | Pulangan yang dijangka bagi setiap trajektori pelaburan. Diselaraskan mengikut indeks.                           | `[0.15, 0.10]`               |
| rest_breaches           | list[float]    | Penyelewengan kekangan maksimum dalam setiap trajektori pelaburan. Diselaraskan mengikut indeks.                 | `[0.0, 0.25]`                |
| transaction_costs       | list[float]    | Anggaran kos transaksi yang berkaitan dengan setiap trajektori pelaburan. Diselaraskan mengikut indeks.          | `[0.01, 0.02]`               |
# Mula Guna
Sahkan diri menggunakan [kunci API](https://quantum.cloud.ibm.com/) anda dan pilih Fungsi Qiskit seperti berikut. (Coretan ini mengandaikan anda sudah [menyimpan akaun anda](/guides/functions#install-qiskit-functions-catalog-client) ke persekitaran setempat anda.)

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

## Contoh: Pengoptimuman portfolio dinamik dengan tujuh aset
Contoh ini menunjukkan cara melaksanakan fungsi pengoptimuman portfolio dinamik (DPO) dan melaraskan tetapannya untuk prestasi optimum. Ia merangkumi langkah-langkah terperinci untuk memperhalusi parameter bagi mencapai hasil yang diingini.

Kes ini melibatkan tujuh aset, empat langkah masa, dan empat qubit resolusi, menghasilkan keperluan keseluruhan sebanyak 112 qubit.
### 1. Baca aset yang disertakan dalam portfolio.
Jika semua aset dalam portfolio disimpan dalam folder pada laluan tertentu, anda boleh memuatkannya ke dalam `pandas.DataFrame` dan menukarnya kepada objek format `dict` menggunakan fungsi berikut.

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

Untuk contoh ini, kami telah menggunakan aset [8801.T](https://finance.yahoo.com/quote/8801.T), [CLF](https://finance.yahoo.com/quote/CLF), [GBPJPY](https://finance.yahoo.com/quote/GBPJPY), [ITX.MC](https://finance.yahoo.com/quote/ITX.MC), [META](https://finance.yahoo.com/quote/META), [TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y), dan [XS2239553048](https://finance.yahoo.com/quote/XS2239553048). Rajah berikut menggambarkan data yang digunakan dalam contoh ini, menunjukkan evolusi harga penutup harian aset dari 1 Januari hingga 1 September 2023.

Dalam contoh ini, untuk memastikan keseragaman merentasi tarikh, kami telah mengisi hari bukan perdagangan dengan harga penutup dari tarikh yang tersedia sebelumnya. Kami melaksanakan langkah ini kerana aset yang dipilih berasal dari pasaran berbeza dengan hari perdagangan yang berbeza-beza, menjadikannya penting untuk menyeragamkan set data bagi konsistensi.
![Visualisasi data sejarah aset](../docs/images/guides/global-data-quantum-optimizer/assets.avif)

### 2. Takrifkan masalah.

Takrifkan spesifikasi masalah dengan mengkonfigurasi parameter dalam kamus `qubo_settings`.

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

### 3. Takrifkan tetapan pengoptimum dan ansatz. (Pilihan)
Secara pilihan takrifkan keperluan khusus untuk proses pengoptimuman, termasuk pemilihan pengoptimum dan parameternya, serta spesifikasi primitif dan konfigurasinya.

Untuk Ansatz Tailored, saiz populasi yang dipilih berdasarkan eksperimen terdahulu yang menunjukkan nilai ini menghasilkan pengoptimuman yang stabil dan cekap.

Dalam kes Ansatz Real Amplitudes, anda boleh mengikuti hubungan linear antara ``population_size`` dan bilangan qubit dalam Circuit. Sebagai peraturan anggaran, adalah disyorkan untuk menggunakan **minimum** ``population_size ~ 0.8 * n_qubits`` untuk ansatz `real_amplitudes`.

Dijangkakan bahawa Optimized Real Amplitudes akan mempunyai prestasi pengoptimuman yang lebih baik daripada ansatz Real Amplitudes. Walau bagaimanapun, bilangan pemboleh ubah yang perlu dioptimumkan dalam ansatz ini meningkat jauh lebih pantas berbanding kes Real Amplitudes (lihat [manuskrip](https://arxiv.org/pdf/2412.19150)). Oleh itu, untuk masalah besar, Optimized Real Amplitudes memerlukan lebih banyak pelaksanaan Circuit. Optimized Real Amplitudes berkemungkinan berguna untuk masalah yang memerlukan hingga 100 qubit, tetapi adalah disyorkan untuk berhati-hati semasa menetapkan parameter ``population_size``. Sebagai contoh peningkatan skala dalam ``population_size``, jadual sebelumnya menunjukkan bahawa untuk masalah 84 qubit, Optimized Real Amplitudes memerlukan ``population_size`` 120, manakala untuk masalah 56 qubit, ``population_size`` 40 sudah mencukupi.

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

Anda juga boleh memilih ansatz tertentu. Contoh berikut menggunakan ansatz ``'Tailored'``.

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 4. Jalankan masalah.

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

### 5. Dapatkan semula keputusan.
Seperti yang dinyatakan dalam bahagian [Output](#output), fungsi mengembalikan kamus dengan trajektori pelaburan yang disusun dari terendah ke tertinggi mengikut nilai fungsi objektifnya. Set hasil ini membolehkan pengenalpastian trajektori dengan kos terendah dan penilaian pelaburan yang sepadan. Selain itu, ia membolehkan analisis trajektori berbeza, memudahkan pemilihan yang paling sesuai dengan keperluan atau objektif tertentu. Fleksibiliti ini memastikan pilihan dapat disesuaikan untuk memenuhi pelbagai keutamaan atau senario.
Mulakan dengan mempersembahkan strategi hasil yang mencapai kos objektif terendah yang ditemui semasa proses.

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


### 6. Performance analysis

Last, analyze the performance of your optimization application. Specifically, compare your results, obtained in the previous example, against a random baseline to assess the effectiveness of our approach. If the quantum algorithm demonstrably and consistently produces results with lower cost values, it indicates an effective optimization process.

The figure presents the probability distributions of the objective costs. To generate these distributions, take the list of objective costs from the function result and count the occurrences of each cost value (values rounded to the second decimal place). Then, update the count column accordingly by joining counts of identical rounded values. Note that, for better visual comparison, the occurrence counts have been normalized so that each distribution is displayed between 0 and 1.

![Visualization of the solution of the optimization](../docs/images/guides/global-data-quantum-optimizer/cost_distribution.svg)

As shown in the figure (blue solid line), the cost distribution for our Variational Quantum Eigensolver (post-processed with SQD) approach is sharply concentrated at lower objective cost values, indicating good optimization performance. In contrast, the noisy baseline exhibits a broader distribution, centered around higher cost values. The gray dashed vertical line represents the mean value of the random distribution, further highlighting the consistency of the function in returning optimized investment strategies. For an additional comparison, the black dashed line in the figure corresponds to the solution obtained with the Gurobi optimizer (free version). All these results are further explored in the benchmarks below for the "Mixed Assets" example evaluated with the "Tailored" ansatz.

# Benchmarks

This function was tested under different configurations of resolution qubits, ansatz circuits, and groupings of assets from various sectors: a mix of different assets (Set 1), oil derivatives (Set 2), and IBEX35 (Set 3). See more details in the following table.

| Set       | Date       | Assets                                                                 |
|-----------|------------|------------------------------------------------------------------------|
| **Set 1** | 01/01/2023 | 8801.T, CL=F, GBPJPY=X, ITX.MC, META, TMBMKDE-10Y, XS2239553048         |
| **Set 2** | 01/06/2023 | CL=F, BZ=F, HO=F, NG=F, XOM, RB=F, 2222.SR                               |
| **Set 3** | 01/11/2022 | ACS.MC, ITX.MC, FER.MC, ELE.MC, SCYR.MC, AENA.MC, AMS.MC                |

Two key metrics were used to evaluate solution quality.
1. The objective cost, which measures optimization efficiency by comparing the cost function value from each experiment with results from Gurobi (free version).
2. The Sharpe ratio, which captures the risk-adjusted return of each portfolio, offering insight into the financial performance of the solutions.

Together, these metrics benchmark both computational and financial aspects of the quantum-generated portfolios.


| Example             | Qubits | Ansatz                  | Depth | Runtime Usage (s) | Total usage (s) | Objective cost | Sharpe | Gurobi objective cost | Gurobi Sharpe |
|-------------------------------|--------|-------------------------------|--------|-------------------|------------------|----------------|--------|------------------------|----------------|
| Mixed Assets (Set 1, 4 time steps, 4-bit)   | 112    | Tailored                      | 83     | 12735             | 13095            | -3.78          | 24.82  | -4.25                  | 24.71          |
| Mixed Assets (Set 1,4 time steps, 4 time steps, 4-bit)   | 112    | Real Amplitudes               | 359    | 11739             | 11903            | -3.39          | 23.64  | -4.25                  | 24.71          |
| Oil Derivatives (Set 2, 4 time steps, 3-bit)| 84     | Optimized Real Amplitudes     | 78     | 6180              | 6350             | -3.73          | 19.13  | -4.19                  | 21.71          |
| IBEX35 (Set 3, 4 time steps, 2-bit)         | 56     | Optimized Real Amplitudes     | 96     | 3314              | 3523             | -3.67          | 14.48  | -4.11                  | 16.44          |



Results show that the quantum optimizer, with problem-specific ansatzes, effectively identifies efficient investment strategies across various portfolio types.

Below we detail both the population size and the number of generations specified in the `optimizer_options` dictionary. All other parameters were set to their default values.

| **Example**                | ``population_size`` | ``num_generations``   |
|----------------------------|----------------------|----------------------|
| Mixed Assets Portfolio     | 90                   | 20                   |
| Mixed Assets Portfolio     | 92                   | 20                   |
| Oil Derivatives Portfolio  | 120                  | 20                   |
| IBEX35 Portfolio           | 40                   | 20                   |

The number of generations was set to 20, as this value was found to be sufficient to reach convergence. Additionally, the default values for the optimizer's internal parameters were left unchanged, as they consistently provided good performance and are generally recommended by the literature and implementation guidelines.

# Get support
If you need help, you can send an email to qpo.support@globaldataquantum.com. In your message, provide the function job ID.

## Next steps

<Admonition type="note" title="Recommendations">
*   Read [the associated research paper](https://arxiv.org/pdf/2412.19150).
*   Request access to the function by filling in [this form](https://www.globaldataquantum.com/en/quantum-portfolio-optimizer/#form).
*   Try the [Dynamic Portfolio Optimization](/docs/tutorials/global-data-quantum-optimizer) tutorial.

</Admonition>